# ML Models 03 - Time Series / Walk-Forward CV

**Cel:** Walk-forward cross validation i opcjonalnie Prophet.

**DATA GATE:** Ten notebook wymaga >= 20 snapshotow.
Pelne CV (t+7) mozliwe od ok. 2026-07-11 (>= 37 snapshotow).

**UWAGA:** Jesli nie masz jeszcze 20 snapshotow -- przejdz do notebooka 04 (SHAP/Optuna).
Mozesz wrocic tutaj pozniej.

**Co to jest walk-forward CV:**
Trenuj na danych do dnia T, testuj na T+7. Potem przesun okno o 1 dzien.
NIE uzywaj random split dla danych czasowych -- to byloby leakage!

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parents[1]))

In [2]:
import duckdb
import matplotlib.pyplot as plt
import sys

sys.path.insert(0, "..")

DB_PATH = "../../data/gold/cards.duckdb"
conn = duckdb.connect(DB_PATH, read_only=True)

snapshots = conn.execute(
    "SELECT DISTINCT snapshot_date FROM gold_price_features ORDER BY snapshot_date"
).df()
print(f"Available snapshots: {len(snapshots)}")
print(snapshots["snapshot_date"].to_string())

Available snapshots: 36
0     2026-05-26
1     2026-05-27
2     2026-05-28
3     2026-05-29
4     2026-05-30
5     2026-05-31
6     2026-06-01
7     2026-06-02
8     2026-06-03
9     2026-06-04
10    2026-06-05
11    2026-06-06
12    2026-06-07
13    2026-06-08
14    2026-06-09
15    2026-06-11
16    2026-06-12
17    2026-06-13
18    2026-06-14
19    2026-06-15
20    2026-06-16
21    2026-06-17
22    2026-06-18
23    2026-06-19
24    2026-06-20
25    2026-06-21
26    2026-06-22
27    2026-06-23
28    2026-06-29
29    2026-07-01
30    2026-07-02
31    2026-07-05
32    2026-07-06
33    2026-07-07
34    2026-07-08
35    2026-07-09


## 1. Walk-Forward Cross Validation

Importuj z `src/ml/training/trainer.py`:
```python
from src.ml.training.trainer import walk_forward_cv, InsufficientDataError
```

Jesli dostaniesz `InsufficientDataError` -- za malo danych, wroc pozniej.

In [3]:
from src.ml.training.trainer import (
    walk_forward_cv,
    get_available_snapshots,
    InsufficientDataError,
)
from src.ml.models.lightgbm_model import LightGBMPriceModel, LightGBMParams
from src.ml.training.tracking import setup_experiment, start_run, log_cv_results

try:
    available = get_available_snapshots(conn)
    print(f"Available snapshots: {len(available)}")

    params = LightGBMParams()
    model = LightGBMPriceModel(params)

    setup_experiment()
    with start_run(run_name="walk_forward_cv_nb03") as run:
        # folds=None → auto-generated from available snapshots
        # Raises InsufficientDataError when < 3 folds can be produced.
        cv_df = walk_forward_cv(conn, model)
        log_cv_results(cv_df)
        print(f"MLflow run_id: {run.info.run_id}")

    print(cv_df.to_string())

except InsufficientDataError as e:
    print(f"DATA GATE: {e}")
    print("Come back when you have more snapshots.")
    cv_df = None

C:\Workspace\AviariumSoftware\aviarium.columbarius\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Available snapshots: 36


DATA GATE: Only 2 fold(s) generated (minimum 3 required). Walk-forward CV unlocks at approximately 2026-07-15. Current data spans 2026-05-26 to 2026-07-09.
Come back when you have more snapshots.


## 2. Wizualizacja Wynikow CV

Sprawdz czy MAE jest stabilne w czasie czy sie pogarsza na pozniejszych danych.

In [4]:
if cv_df is not None and not cv_df.empty:
    # Aggregate across tiers per fold — Tier 1 dominates (>99% of cards),
    # so the per-fold mean closely tracks Tier 1 performance.
    fold_summary = (
        cv_df.groupby(["fold_idx", "val_snapshot"])[["mae", "mape"]]
        .mean()
        .reset_index()
        .sort_values("fold_idx")
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    fold_summary.plot(x="val_snapshot", y="mae", ax=axes[0], marker="o")
    axes[0].set_title("MAE per fold (lower is better)")
    axes[0].axhline(
        fold_summary["mae"].mean(), color="red", linestyle="--", label="mean"
    )
    axes[0].legend()

    fold_summary.plot(
        x="val_snapshot", y="mape", ax=axes[1], marker="o", color="orange"
    )
    axes[1].set_title("MAPE per fold (%)")
    axes[1].axhline(
        fold_summary["mape"].mean(), color="red", linestyle="--", label="mean"
    )
    axes[1].legend()

    plt.tight_layout()
    plt.show()
    print(f"Mean MAE:  {fold_summary['mae'].mean():.4f}")
    print(f"Mean MAPE: {fold_summary['mape'].mean():.2f}%")
    print("\nFull per-tier breakdown:")
    print(cv_df.to_string())

## 3. (Opcjonalnie) Prophet dla Wybranych Kart

**Prophet** dziala dobrze dla kart z dluga historia (>50 punktow cen).
Jest OPCJONALNY -- jesli LightGBM wystarczy, nie potrzebujesz go.

Instalacja: `uv add prophet`

Jak to zrobic:
1. Wybierz kilka kart Tier 2 z dluga historia
2. Przygotuj DataFrame z kolumnami 'ds' (data) i 'y' (cena)
3. Porownaj MAE Prophet vs LightGBM na tych samych kartach

In [5]:
# Optional — uncomment to test Prophet for selected cards.
# Requires: uv add prophet
#
# from prophet import Prophet
#
# card_uuid = 'CARD_UUID_HERE'
# prices = conn.execute(
#     "SELECT snapshot_date AS ds, eur AS y "
#     "FROM gold_price_features WHERE uuid = ?",
#     [card_uuid],
# ).df()
# prices['ds'] = pd.to_datetime(prices['ds'])
#
# m = Prophet()
# m.fit(prices)
# future = m.make_future_dataframe(periods=7)
# forecast = m.predict(future)
# m.plot(forecast)
print("Prophet optional — uncomment the code above to test it.")

Prophet optional — uncomment the code above to test it.


## Observations

- **Number of CV folds:** Data-gated — only 2 folds could be generated (minimum 3 required by `walk_forward_cv`). `InsufficientDataError` was raised: "Only 2 fold(s) generated (minimum 3 required). Walk-forward CV unlocks at approximately 2026-07-15. Current data spans 2026-05-26 to 2026-07-09."
- **Mean MAE across folds / Mean MAPE:** Not computed — `cv_df` is `None` because the CV gate was not met, so cell 2 ("Wizualizacja Wynikow CV") produced no output.
- **Is MAE stable over time or degrading on later folds:** Not assessable yet — no folds were produced.
- **Prophet better than LightGBM for Tier 2 cards:** Not tested — the optional Prophet cell was left commented out (per its own instructions, since LightGBM CV itself is still gated).
- **Conclusions:** This notebook remains genuinely data-gated, same as previously documented. 36 snapshots are available (2026-05-26 → 2026-07-09), but `walk_forward_cv` needs enough distinct dates to produce ≥ 3 folds, which per the raised error will not happen until approximately **2026-07-15**. Come back after that date to run walk-forward CV and (optionally) the Prophet comparison.
